In [16]:
!pip install convokit

from convokit import Corpus, download

corpus = Corpus(filename=download("friends-corpus"))
corpus.print_summary_stats()

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 16.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 610.9/610.9 kB 11.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 19.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 44.2 MB/s  0:00:00
  Created wheel for convokit: filename=convokit-5.0.0-py3-none-any.whl size=404788 sha256=a69434bbd913cb1ae10aeae50b0a9fdd14f92d8bcc9550cb2ee7443f09a6af88
  Stored in directory: /root/.cache/pip/wheels/3c/a6/d0/b15d166a32d56d7c6f9463f6294f658c8eec7a4f7ee2920444
Successfully built convokit
  Attempting uninstall: h5py
    Found existing installation: h5py 3.14.0
    Uninstalling h5py-3.14.0:
      Successfully uninstalled h5py-3.14.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11/11 [convokit]
No configuration file found at /root/.convokit/config.yml; writin

In [17]:
A = "Rachel Green"
B = "Ross Geller"
C = "Monica Geller"

pair_AB = {A, B}
pair_AC = {A, C}

print("Person A:", A)
print("Pair A+B:", pair_AB)
print("Pair A+C:", pair_AC)

Person A: Rachel Green
Pair A+B: {'Ross Geller', 'Rachel Green'}
Pair A+C: {'Rachel Green', 'Monica Geller'}


In [18]:
from collections import Counter

speaker_counts = Counter(
    utt.speaker.id
    for utt in corpus.iter_utterances()
)

for name in [A, B, C]:
    print(f"{name}: {speaker_counts[name]} utterances")

Rachel Green: 9331 utterances
Ross Geller: 9161 utterances
Monica Geller: 8498 utterances


In [22]:
from collections import Counter

def extract_pair_segments(
    corpus,
    pair,
    min_turns=4,
    min_turns_per_person=2
):
    """
    Extract contiguous dialogue segments involving only a chosen pair.

    A segment ends whenever someone outside the pair speaks.
    Both characters must speak enough for the segment to be retained.
    """
    segments = []

    for convo in corpus.iter_conversations():
        current_segment = []

        def save_segment():
            # Ignore short segments
            if len(current_segment) < min_turns:
                return

            # Count turns from each character
            counts = Counter(turn["speaker"] for turn in current_segment)

            # Require both characters to participate
            if any(counts[person] < min_turns_per_person for person in pair):
                return

            segments.append({
                "conversation_id": convo.id,
                "pair": sorted(pair),
                "turns": current_segment.copy()
            })

        for utt in convo.iter_utterances():
            speaker = utt.speaker.id

            if speaker in pair:
                # Add this utterance to the current pair-only segment
                current_segment.append({
                    "speaker": speaker,
                    "text": utt.text
                })
            else:
                # A third person spoke, so close the current segment
                save_segment()
                current_segment = []

        # Don't forget the final segment in the scene
        save_segment()

    return segments


# Extract dialogue segments for both relationships
scenes_AB = extract_pair_segments(corpus, pair_AB)
scenes_AC = extract_pair_segments(corpus, pair_AC)

print(f"Rachel–Ross segments: {len(scenes_AB)}")
print(f"Rachel–Monica segments: {len(scenes_AC)}")

Rachel–Ross segments: 481
Rachel–Monica segments: 295


In [29]:
def show_scene(scene):
    print("Conversation ID:", scene["conversation_id"])
    print("Pair:", " + ".join(scene["pair"]))
    print("-" * 60)

    for turn in scene["turns"]:
        print(f'{turn["speaker"]}: {turn["text"]}')


# Display the first 3 Rachel–Ross segments
for i, scene in enumerate(scenes_AB[:3]):
    print(f"\n=== Rachel–Ross segment {i + 1} ===")
    show_scene(scene)

# Display the first 3 Rachel–Monica segments
for i, scene in enumerate(scenes_AC[:3]):
    print(f"\n=== Rachel–Monica segment {i + 1} ===")
    show_scene(scene)


=== Rachel–Ross segment 1 ===
Conversation ID: s01_e01_c14_u001
Pair: Rachel Green + Ross Geller
------------------------------------------------------------
Ross Geller: Mmm. Oh, no-
Rachel Green: Sorry-
Ross Geller: No no no, go-
Rachel Green: No, you have it, really, I don't want it-
Ross Geller: Split it?
Rachel Green: Okay.
Ross Geller: Okay. You know you probably didn't know this, but back in high school, I had a, um, major crush on you.
Rachel Green: I knew.
Ross Geller: You did! Oh.... I always figured you just thought I was Monica's geeky older brother.
Rachel Green: I did.
Ross Geller: Oh. Listen, do you think- and try not to let my intense vulnerability become any kind of a factor here- but do you think it would be okay if I asked you out? Sometime? Maybe?
Rachel Green: Yeah, maybe...
Ross Geller: Okay... okay, maybe I will...
Rachel Green: Goodnight.
Ross Geller: Goodnight.

=== Rachel–Ross segment 2 ===
Conversation ID: s01_e02_c05_u001
Pair: Rachel Green + Ross Geller
--

In [28]:
!pip -q install spacy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 21.6 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [38]:
import spacy
import re
import copy
import json

nlp = spacy.load("en_core_web_sm")

# Fixed labels for the three main characters
MAIN_NAMES = {
    "rachel green": "Person A",
    "rachel": "Person A",
    "rach": "Person A",
    "rach-rach": "Person A",
    "ross geller": "Person B",
    "ross": "Person B",
    "rrrooossss": "Person B",
    "monica geller": "Person C",
    "monica": "Person C",
}

# Common words and phrases that should NOT be treated as names
FALSE_POSITIVES = {
    "y'know", "ya know", "ya", "wha", "whoah", "nono",
    "back", "lemme", "uhhhooo", "oohhh", "wooo", "umm",
    "uh", "um", "oh", "god", "dude", "sweet", "honey",
    "hurry", "bang", "kick", "ball", "big", "la", "se",
    "nah", "sup", "wanna", "allright", "okidoki",
    "surprised", "momentary", "twins", "uncle", "dad",
    "mom", "momma", "potato head", "easy bake oven",
    "saran wrap", "spray lysol", "baby got back",
    "jurassic parka", "curious george"
}

In [39]:
def get_label(index):
    """Convert 0, 1, ... into A, B, ... Z, AA, AB, etc."""
    label = ""
    while index >= 0:
        label = chr(ord("A") + index % 26) + label
        index = index // 26 - 1
    return f"Person {label}"


def anonymize_text(text, other_names):
    doc = nlp(text)
    replacements = []

    # Replace main-character names, checking longer names first
    for name, label in sorted(
        MAIN_NAMES.items(), key=lambda item: len(item[0]), reverse=True
    ):
        pattern = re.compile(
            rf"(?<!\w){re.escape(name)}(?!\w)",
            re.IGNORECASE
        )
        for match in pattern.finditer(text):
            replacements.append(
                (match.start(), match.end(), label)
            )

    # Detect other people's names
    for ent in doc.ents:
        if ent.label_ != "PERSON":
            continue

        name = ent.text.strip()
        if name.lower() in FALSE_POSITIVES or len(name) <= 2:
            continue

        # Don't overwrite a main-character name
        overlaps = any(
            start < ent.end_char and end > ent.start_char
            for start, end, _ in replacements
        )
        if overlaps:
            continue

        key = name.lower()

        if key not in other_names:
            other_names[key] = get_label(len(other_names) + 3)

        replacements.append(
            (ent.start_char, ent.end_char, other_names[key])
        )

    # Apply replacements from right to left
    replacements.sort(key=lambda item: item[0], reverse=True)
    result = text
    last_start = len(text) + 1

    for start, end, label in replacements:
        if end > last_start:
            continue
        result = result[:start] + label + result[end:]
        last_start = start

    return result

In [40]:
def anonymize_scenes(scenes):
    anonymized = copy.deepcopy(scenes)

    for scene in anonymized:
        # Reset additional-name mappings for each segment
        other_names = {}

        for turn in scene["turns"]:
            original_speaker = turn["speaker"]
            normalized = original_speaker.lower().strip()

            # Map main characters to A/B/C.
            # Give any other speaker a label starting at D.
            if normalized in MAIN_NAMES:
                turn["speaker"] = MAIN_NAMES[normalized]
            else:
                key = normalized
                if key not in other_names:
                    other_names[key] = get_label(len(other_names) + 3)
                turn["speaker"] = other_names[key]

            # Replace names mentioned inside the dialogue
            turn["text"] = anonymize_text(
                turn["text"], other_names
            )

        # Anonymize the pair metadata
        scene["pair"] = [
            MAIN_NAMES.get(person.lower().strip(), person)
            for person in scene["pair"]
        ]

    return anonymized


# Always start from the original extracted datasets
anon_AB = anonymize_scenes(scenes_AB)
anon_AC = anonymize_scenes(scenes_AC)

print("Rachel–Ross segments:", len(anon_AB))
print("Rachel–Monica segments:", len(anon_AC))

Rachel–Ross segments: 481
Rachel–Monica segments: 295


In [43]:
# Inspect examples before saving
for label, dataset in [
    ("Rachel–Ross", anon_AB),
    ("Rachel–Monica", anon_AC)
]:
    print(f"\n=== {label} ===")

    for scene in dataset[20:25]:
        print("\nScene:", scene["conversation_id"])
        for turn in scene["turns"]:
            print(f'{turn["speaker"]}: {turn["text"]}')



=== Rachel–Ross ===

Scene: s01_e16_c02_u001
Person A: So, is this just gonna be you and Person D?
Person B: No, Susans gonna be there too. Weve got dads, weve got lesbians, the whole parenting team.
Person A: Well, isnt, isnt that gonna be weird?
Person B: No, no. I mean, it mighta been at first, but by now I, I think I'm pretty comfortable with the whole situation.

Scene: s01_e16_c13_u001
Person B: Im gonna be a father.
Person A: This is just occurring to you?
Person B: I always knew I was havin a baby, I just never realised the baby was having me.
Person A: Oh, youre gonna be great!
Person B: Aw, how can you say that? I cant even get Marcel to stop eating the bath mat. How am I gonna raise a kid?

Scene: s01_e18_c01_u001
Person B: Look, I am totally, totally over her, OK, I just... : Hiiii!
Person A: Hi! How are you?
Person B: We're fine, we're fine.
Person A: OK.

Scene: s01_e18_c02_u001
Person B: No. No, uh, Phoebs? You can'tyou can't do...
Person A: Oh wait, I have the ten of s

In [44]:
import json

with open("friends_AB_anonymized.json", "w", encoding="utf-8") as f:
    json.dump(anon_AB, f, ensure_ascii=False, indent=2)

with open("friends_AC_anonymized.json", "w", encoding="utf-8") as f:
    json.dump(anon_AC, f, ensure_ascii=False, indent=2)

print("Anonymized datasets saved!")

Anonymized datasets saved!


In [45]:
import json
import random
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)

# Load the anonymized datasets
with open("/content/friends_AB_anonymized.json", encoding="utf-8") as f:
    scenes_AB = json.load(f)

with open("/content/friends_AC_anonymized.json", encoding="utf-8") as f:
    scenes_AC = json.load(f)

print("A-B segments:", len(scenes_AB))
print("A-C segments:", len(scenes_AC))

# Collect unique episode/conversation IDs across BOTH datasets
all_scenes = scenes_AB + scenes_AC
episode_ids = sorted({
    scene["conversation_id"] for scene in all_scenes
})

train_ids, test_ids = train_test_split(
    episode_ids,
    test_size=0.20,
    random_state=SEED
)

train_ids = set(train_ids)
test_ids = set(test_ids)

# Split both datasets using the same episode-level split
train_AB = [s for s in scenes_AB if s["conversation_id"] in train_ids]
test_AB  = [s for s in scenes_AB if s["conversation_id"] in test_ids]

train_AC = [s for s in scenes_AC if s["conversation_id"] in train_ids]
test_AC  = [s for s in scenes_AC if s["conversation_id"] in test_ids]

print("\nTraining A-B segments:", len(train_AB))
print("Testing A-B segments:", len(test_AB))
print("Training A-C segments:", len(train_AC))
print("Testing A-C segments:", len(test_AC))

# Verify that no episode appears in both splits
assert train_ids.isdisjoint(test_ids)

# Save the split so it can be reused consistently
with open("/content/friends_split.json", "w", encoding="utf-8") as f:
    json.dump({
        "train_episode_ids": sorted(train_ids),
        "test_episode_ids": sorted(test_ids)
    }, f, indent=2)

print("\nEpisode-level split saved.")

A-B segments: 481
A-C segments: 295

Training A-B segments: 383
Testing A-B segments: 98
Training A-C segments: 240
Testing A-C segments: 55

Episode-level split saved.
